In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
stops = pd.read_excel('/Users/Tuins/OneDrive - Delft University of Technology/Y3 Minor RUG/Advanced data analysis/000 group project/Relevent stops.xlsx')
trips = pd.read_csv('/Users/Tuins/OneDrive - Delft University of Technology/Y3 Minor RUG/Advanced data analysis/000 group project/gtfs-nl/trips.txt')
routes = pd.read_csv('/Users/Tuins/OneDrive - Delft University of Technology/Y3 Minor RUG/Advanced data analysis/000 group project/gtfs-nl/routes.txt')
cal = pd.read_csv('/Users/Tuins/OneDrive - Delft University of Technology/Y3 Minor RUG/Advanced data analysis/000 group project/gtfs-nl/calendar_dates.txt')
st = pd.read_csv('/Users/Tuins/OneDrive - Delft University of Technology/Y3 Minor RUG/Advanced data analysis/000 group project/gtfs-nl/stop_times.txt', usecols=["trip_id", "stop_id", "departure_time"],
                 dtype={"stop_id": str})

First, we select only the columns we need from the files:

In [3]:
stops['station_id'] = stops['parent_station'].fillna(stops['stop_id'])
stops = stops[['station_id', 'parent_station', 'stop_name', 'stop_id', 'stop_lat', 'stop_lon']]
display(stops.head())

,station_id,parent_station,stop_name,stop_id,stop_lat,stop_lon
0,4185704,NaN,"Opende, Topweer",4185704,53.175081,6.190373
1,4185703,NaN,"Opende, Topweer",4185703,53.174522,6.192111
2,4185706,NaN,"Opende, Verbindingsweg",4185706,53.174197,6.197270
3,4185705,NaN,"Opende, Verbindingsweg",4185705,53.174056,6.197713
4,stoparea:613323,stoparea:613323,"Opende, Sjallemastraat",4183013,53.174134,6.200817


In [4]:
trips = trips[['route_id', 'service_id', 'trip_id']]
display(trips.head())

,route_id,service_id,trip_id
0,17562,1687,260620735
1,17562,144,260620765
2,17562,1502,260620801
3,17562,1498,260620844
4,17562,627,260623502


In [5]:
routes = routes[['route_type', 'route_id']]
display(routes.head())

,route_type,route_id
0,3,134842
1,3,134844
2,3,134845
3,3,134846
4,3,134847


In [6]:
calender = cal[['date', 'service_id']]
display(calender.head())

,date,service_id
0,20261002,1
1,20261002,2
2,20261211,2
3,20261002,3
4,20261101,3


In [7]:
st = st[['trip_id', 'stop_id', 'departure_time']]
display(st.head())

,trip_id,stop_id,departure_time
0,260620735,2993758,10:20:00
1,260620735,2993739,10:28:00
2,260620735,2992835,10:39:00
3,260620735,2993990,10:57:00
4,260620765,2993758,07:50:00


Then, we merged all the tables into 1 big one

In [8]:
stops_st = stops.merge(st, on='stop_id')

stops_st_trips = stops_st.merge(trips, on='trip_id')
stops_st_trips_routes = stops_st_trips.merge(routes, on='route_id')
stops_st_trips_routes_calender = stops_st_trips_routes.merge(calender, on='service_id')

merged = stops_st_trips_routes_calender
display(merged.head())

,station_id,parent_station,stop_name,stop_id,stop_lat,stop_lon,trip_id,departure_time,route_id,service_id,route_type,date
0,4185704,NaN,"Opende, Topweer",4185704,53.175081,6.190373,386887212,07:24:00,151973,3046,3,20261022
1,4185704,NaN,"Opende, Topweer",4185704,53.175081,6.190373,386887212,07:24:00,151973,3046,3,20261023
2,4185704,NaN,"Opende, Topweer",4185704,53.175081,6.190373,386887212,07:24:00,151973,3046,3,20261029
3,4185704,NaN,"Opende, Topweer",4185704,53.175081,6.190373,386887212,07:24:00,151973,3046,3,20261030
4,4185704,NaN,"Opende, Topweer",4185704,53.175081,6.190373,386887212,07:24:00,151973,3046,3,20261105


then, we filter the table for only the rail (type = 2)

In [9]:
merged = merged[merged['route_type'] == 2]
#merged.head()

then, we filter it on a day of the week ( Filtered on Tuesday 13 oktober 2026) and the times (between 6:30 and 9:00)

In [10]:
day = 20261013
merged = merged[merged['date'] == day]
#merged.head()

In [11]:
merged = merged[(merged["departure_time"] >= "06:30:00") & (merged["departure_time"] <= "09:00:00")]
merged.head()

,station_id,parent_station,stop_name,stop_id,stop_lat,stop_lon,trip_id,departure_time,route_id,service_id,route_type,date
234169,stoparea:17912,stoparea:17912,Baflo,2992304,53.360989,6.518723,313314500,08:09:00,139849,689,2,20261013
234219,stoparea:17912,stoparea:17912,Baflo,2992304,53.360989,6.518723,313316252,07:39:00,139849,689,2,20261013
234880,stoparea:17912,stoparea:17912,Baflo,2992304,53.360989,6.518723,338805585,07:20:00,139849,689,2,20261013
235490,stoparea:17912,stoparea:17912,Baflo,2992304,53.360989,6.518723,380547487,06:50:00,139849,1750,2,20261013
235539,stoparea:17912,stoparea:17912,Baflo,2992304,53.360989,6.518723,380547525,08:50:00,139849,1750,2,20261013


Then, I deleted the columns we do not need anymore

In [12]:
small = merged[['stop_name', 'station_id', 'stop_lat', 'stop_lon', 'trip_id', 'departure_time']]
small = small.drop_duplicates(subset=["station_id", "trip_id"])

display(small.head())

,stop_name,station_id,stop_lat,stop_lon,trip_id,departure_time
234169,Baflo,stoparea:17912,53.360989,6.518723,313314500,08:09:00
234219,Baflo,stoparea:17912,53.360989,6.518723,313316252,07:39:00
234880,Baflo,stoparea:17912,53.360989,6.518723,338805585,07:20:00
235490,Baflo,stoparea:17912,53.360989,6.518723,380547487,06:50:00
235539,Baflo,stoparea:17912,53.360989,6.518723,380547525,08:50:00


Then I filtered the table again and I calculated the frequency

In [13]:
stations = (small.groupby("station_id")
            .agg(stop_name=('stop_name', 'first'),
                 trains_per_h=("trip_id", lambda s: s.nunique() / 2.5),
                 stop_lat=("stop_lat", "mean"),
                 stop_lon=("stop_lon", "mean"),)
            .reset_index())

display(stations.head())


,station_id,stop_name,trains_per_h,stop_lat,stop_lon
0,stoparea:153154,De Westereen,4.0,53.248281,6.035517
1,stoparea:165459,Dronryp,4.8,53.177731,5.634844
2,stoparea:17789,Hurdegaryp,4.4,53.218955,5.934652
3,stoparea:17803,Grou-Jirnsum,4.0,53.089619,5.822405
4,stoparea:17807,Grijpskerk,4.0,53.255805,6.310047


Lastly, we calculated the SWT and AWT and added the columns

In [14]:
stations["SWT"] = 0.5 * (60 / stations["trains_per_h"])
stations["AWT"] = stations["SWT"] + 0.75

display(stations.head())

,station_id,stop_name,trains_per_h,stop_lat,stop_lon,SWT,AWT
0,stoparea:153154,De Westereen,4.0,53.248281,6.035517,7.500000,8.250000
1,stoparea:165459,Dronryp,4.8,53.177731,5.634844,6.250000,7.000000
2,stoparea:17789,Hurdegaryp,4.4,53.218955,5.934652,6.818182,7.568182
3,stoparea:17803,Grou-Jirnsum,4.0,53.089619,5.822405,7.500000,8.250000
4,stoparea:17807,Grijpskerk,4.0,53.255805,6.310047,7.500000,8.250000


In [15]:
stations.to_csv("stations_trains_per_h.csv", index=False)